In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as func

In [ ]:
torch

In [ ]:
with open("input.txt","r") as f:
    txt = f.read()
len(txt)
txt[:1000]

In [ ]:
chars = sorted(list(set(txt)))
vocab_size = len(chars)
"".join(chars)

In [ ]:
stoi = {char:n for n,char in enumerate(chars)}
itos = {n:char for n,char in enumerate(chars)}
encode = lambda x: [stoi[i] for i in x]
decode = lambda x: "".join([itos[i] for i in x])
decode(encode("hii encode"))

In [ ]:
itos

In [ ]:
encode(txt)

In [ ]:
import torch
data = torch.tensor(encode(txt),dtype=torch.long)
data.shape,data.dtype
data[:100]

In [ ]:
n = int(0.9*len(data))
training_data = data[:n]
validation_data = data[n:]

In [ ]:
block_size = 256
x = training_data[:block_size]
y = training_data[1:block_size+1]

for i in range(block_size):
    train_data = x[:i+1]
    test_data = y[i]
    print(train_data,test_data)


In [ ]:
torch.manual_seed(1337)
batch_size = 64
def get_batch(split):
    data = training_data if split=="train" else validation_data
    ix = torch.randint(len(data)-block_size,(batch_size,))
    xb = torch.stack([data[i:i+block_size] for i in ix])
    yb = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return xb, yb

xb, yb = get_batch("train")

for i in range(batch_size):
    for j in range(block_size):
        context = xb[i,:j+1]
        target = yb[i,j]
        print(context,target)

In [ ]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.query = nn.Linear(embedding_size,head_size,bias=False)
        self.keys = nn.Linear(embedding_size,head_size,bias=False)
        self.values = nn.Linear(embedding_size,head_size,bias=False)
        self.register_buffer("tril",torch.tril(torch.ones((block_size,block_size))))
        self.dropout = nn.Dropout(dropout)
    
    def forward(self,data):
        B,T,C = data.shape
        data_query = self.query(data)  #B,T,head_size
        data_key = self.keys(data)     #B,T,head_size

        wei = data_query @ data_key.transpose(-2,-1) * data_key.shape[-1]**(-0.5)  #B,T,head_size @ B,head_size,T -> B,T,T

        a_affinity_masked = wei.masked_fill(self.tril[:T,:T]==0,float('-inf')) #mark future tokens cant communite with current token so -inf.
        a_prob = func.softmax(a_affinity_masked,-1)
        a_prob = self.dropout(a_prob)   # here dropout make some probablity to zero
        data_values = self.values(data)
        data_premean = a_prob@data_values
        return data_premean

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,n_head,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(embedding_size,embedding_size)
        self.dropout = nn.Dropout(dropout)
    def forward(self,data):
        out =  torch.cat([head(data) for head in self.heads],-1)
        out = self.proj(out)
        return self.dropout(out)  # here dropout will prevent some neurons in participate in forward and in backword some neurons will not be updated.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self,n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed,4*n_embed,bias=True),
            nn.ReLU(),
            nn.Linear(4*n_embed,n_embed,bias=True),
            nn.Dropout(dropout)
        )
    
    def forward(self,X):
        return self.net(X)

In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHeadAttention(n_head,embedding_size//n_head)
        self.feed = FeedForward(n_embed=embedding_size)
        self.norm1 = nn.LayerNorm(embedding_size)
        self.norm2 = nn.LayerNorm(embedding_size)

    def forward(self,x):
        x = x + self.sa(self.norm1(x))
        x = x + self.feed(self.norm2(x))
        return x

In [ ]:
torch.manual_seed(1337)
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,embedding_size)
        self.positional_embedding = nn.Embedding(block_size,embedding_size) # both embed size is same as we need to add
        #self.sa = Head(embedding_size)
        self.blocks = nn.Sequential(
            Block(),
            Block(),
            Block(),
            Block(),
            Block(),
            Block(),
            nn.LayerNorm(embedding_size)
        )
        self.im_head = nn.Linear(embedding_size, vocab_size)

    def forward(self,idx,targets=None):
        B,T = idx.shape
        token_embed = self.token_embedding_table(idx)  #(Batch,Time or context_size,Channel or embed dim)
        positional_embed = self.positional_embedding(torch.arange(T))
        total_embed = token_embed + positional_embed
        x = self.blocks(total_embed)
        logits = self.im_head(x) #(Batch,Time or context_size,vocab_size)

        if targets==None:
            loss = None
        else:
            B,T,C = logits.shape
            B,T = targets.shape
            logits = logits.reshape(B*T,C)
            targets = targets.reshape(B*T)
            loss = func.cross_entropy(logits,targets)

        return logits, loss
    
    def generate(self,idx,max_new_tokens=100):
        for _ in range(max_new_tokens):
            idx = idx[:,-block_size:]
            logits, loss = self(idx)
            batch_last_ele = logits[:,-1,:]  # B,C
            prob = func.softmax(batch_last_ele,dim=-1)  # B,C
            idx_next = torch.multinomial(prob,num_samples=1)   #B,1
            idx = torch.cat((idx,idx_next),dim=1) # B, T+1
        return idx

batch_size = 64
block_size = 256
epochs = 5000
lr = 3e-4
embedding_size = 384
n_head = 6
dropout = 0.2

bigram = BigramLanguageModel()
output, loss = bigram(xb,yb)
decode(bigram.generate(idx=torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)[0].tolist())


In [ ]:
optimizer = torch.optim.Adam(bigram.parameters(),lr=lr)

In [ ]:
torch.manual_seed(1337)
for _ in range(epochs):
    print("epoch",_)
    xb, yb = get_batch("train")
    logits, loss = bigram(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
loss.item()

In [ ]:
import pickle
with open('gpt_model.pkl', 'wb') as file:
    pickle.dump(bigram, file)

model = torch.load('gpt_model.pkl')
model.eval()
with open('gpt_model.pkl', 'rb') as handle:
    b = pickle.load(handle)

In [ ]:
torch.zeros((1,1),dtype=torch.long)

In [ ]:
predicted_token = [[0]]

In [ ]:
while(True):
    predicted_token = bigram.generate(idx=torch.tensor(predicted_token,dtype=torch.long),max_new_tokens=1).tolist()
    print(decode([predicted_token[0][-1]]),end="")

In [ ]:
while(True):
    predicted_token = bigram.generate(idx=torch.tensor(predicted_token,dtype=torch.long).reshape(len(predicted_token),2),max_new_tokens=1)
    print(predicted_token)
    #print(decode(predicted_token[-1:]),end="")

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2
data = torch.randn(B,T,C)
data

In [ ]:
data_avg = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        prev_slice = data[b,:t+1]
        data_avg[b,t] = torch.mean(prev_slice,0)

In [ ]:
print(data[0])
print(data_avg[0])

In [ ]:
a_tril = torch.tril(torch.ones((3,3)))
a_tril_avg = a_tril / torch.sum(a_tril,0,keepdim=True)
a_tril, a_tril_avg, torch.sum(a_tril,0,keepdim=True)

In [ ]:
a_tril = torch.tril(torch.ones(T,T))
a_tril = a_tril/ torch.sum(a_tril,1,keepdim=True)  #T,T
data_premean = a_tril @ data # T,T-> B,T,T @ B,T,C -> B,T,C
#print(a_tril, data[1],data_premean[1],data_avg[1])
torch.allclose(data_premean[1],data_avg[1])
data_premean[1]-data_avg[1]

In [ ]:
#OneHead self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32
data = torch.randn(B,T,C)
a_tril = torch.tril(torch.ones((T,T)))
# each position of data have 2 vector query(says what its looking) and keys(what it contains) and values(if some previous token is interesting
#then We will take values information from raw data or emcoded text)
#so we will do dot product of query and keys
head_size = 16
query = nn.Linear(C,head_size,bias=False)
keys = nn.Linear(C,head_size,bias=False)
values = nn.Linear(C,head_size,bias=False) 
data_query = query(data)  #B,T,head_size
data_key = keys(data)     #B,T,head_size

wei = data_query @ data_key.transpose(-2,-1)  #B,T,head_size @ B,head_size,T -> B,T,T

#a_affinity = torch.zeros((T,T)) #affinity
a_affinity_masked = wei.masked_fill(a_tril==0,float('-inf')) #mark future tokens cant communite with current token so -inf.
a_prob = func.softmax(a_affinity_masked,-1)
data_values = values(data)
data_premean = a_prob@data_values
a_prob[0]
# we will use this approach as where we are define 0, there we can use non zero to mark comparatively more affinity of that token withother one.


attention mechanism is a communication mechanism. which tells depending on text and encoding vectors(positional also) how tokens are connected
there is no inter-batch communication, there is only intra-batch communication, so all batches calculation can run independently. 
sometimes in sentiment analysis like usecases we want all token including future and past tokens talk with each other, then we will
delete this line a_affinity_masked = wei.masked_fill(a_tril==0,float('-inf')) to remove restriction of communiaction with future token Then
this is called encoder, our implementation is decoder where we restrict future token and present token communation.


There is a concept called cross-attention where we only use present token to calculate query and we will use external source to calculate keys and values
this is used when we want communication with external source.our thing called self-attention because we use same batch token as keys and values source.

one fact with softmax is if - and + values are vary high, then output sharpens towards maximum makie it like one-hot encoding.
so we want input of softmax's varience is low. 
but in attention calculation if we dont divide with sqrt_root(len(k)), then varience will be in order of head_size.
so we will divide with root of key size and final calculation will be
softmax( (q(data)@k(data)) /root(key_len) )*values(data)

We use add and normalization to optimize multi-blocked self-attention layer. as if we use 5 blocks of self-attentions, then effect of first self-attention on output is minimal
so we add output of each block so that there is a gradient flow from output to directly to first self-attention block.so effect of first block still there.
difference between layer and batch normalization is in batch normalization we do standardization through columns and in batch normlization we do standardization through samples or rows.
in layer normalization we weighted batcNormalization output and add bias to shift also to make scaling more flexible.
(X-mean(X) / root(var(X)+eps)) * gamma + beta
As there is gamma nad beta so output would not be gaussian dist.


generally in paper we use add and norm after multihead and feedforward but now we use prenorm where we do add and norm before going to attention layer and feed forward layer.

In paper there is encoder and decoder model because it used translation as example.In translation first input langage  will be encoded with full direction, then we will pass
target language to decoder which give in only past direction so in decoder past predicted token and input language sentence is given as input to head.

##SQL
To convert from datetime to string use - select to_char(timestamp,'YYYY-MM-DD') from table
To convert from string to datetime use - select date::datetime from table
to fetch month from datetime use - select extract(MONTH from date) from table
to fetch month from datetime use - select date_trunc('month',date) from table